In [ ]:

# RAG (Retrieval-Augmented Generation) main script
"""
1. Run pip install -r requirements.txt before executing this script.
Make sure your: 
version of below are correct. 
langchain==0.1.16 
langchain-community==0.0.38 
openai==0.28.1
"""
from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.chains import RetrievalQA
from langchain.llms import LlamaCpp
from langchain.chat_models import ChatOpenAI


"""
1. Load the document and split it into chunks
"""
print("Loading document...")
loader = PyMuPDFLoader("about-me.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} documents.")


text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=5)
all_splits = text_splitter.split_documents(documents)
print(f"Total chunks created: {len(all_splits)}")

"""
Create a persistent Chroma vector database in the 'db' directory.
Use the MiniLM-L6-v2 embedding model running on CPU for text embeddings.
Convert all document chunks (all_splits) into vector embeddings.
Store embeddings and metadata inside the Chroma database for retrieval.
Print confirmation once the vector store is successfully created.
"""
persist_directory = 'db'
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model_kwargs = {'device': 'cpu'}
embedding = HuggingFaceEmbeddings(model_name=model_name,
                                  model_kwargs=model_kwargs)
vectordb = Chroma.from_documents(documents=all_splits, 
                                 embedding=embedding, 
                                 persist_directory=persist_directory)

"""
Check the contents of the ChromaDB to ensure embeddings are stored correctly
"""
# import sqlite3
# conn = sqlite3.connect("db/chroma.sqlite3")
# cursor = conn.cursor()
# # List all tables
# cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
# tables = cursor.fetchall()
# print("Tables:", tables)


"""
Load a local Llama‑2 7B Chat GGUF model using the LlamaCpp runtime.
Enable GPU acceleration with n_gpu_layers and set batch/context sizes.
Use f16_kv for faster inference and lower memory usage.
Stream generated tokens to the console via CallbackManager.
"""
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

llm = LlamaCpp(
    model_path="/Users/nowshad/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q2_K.gguf",
    n_gpu_layers=100,
    n_batch=512,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
print("LLM initialized.")



"""
Define two prompt templates: one generic and one specifically formatted for Llama models.
The Llama‑optimized prompt uses the <SYS> and [INST] structure required by Llama‑2/3 chat models.
Create a ConditionalPromptSelector that automatically chooses the correct prompt based on the LLM type.
If the active model is LlamaCpp, the system selects the Llama‑style prompt; otherwise it uses the default prompt.
This ensures consistent formatting and prevents prompt‑format errors across different LLM backends.
Retrieve the appropriate prompt for the currently loaded LLM instance.
The final 'prompt' object is then ready to be passed into an LLMChain or used for inference.
"""
from langchain.chains import LLMChain
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

DEFAULT_LLAMA_SEARCH_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""<<SYS>> 
    You are a helpful assistant eager to assist with providing better Google search results.
    <</SYS>> 
    
    [INST] Provide an answer to the following question in 150 words. Ensure that the answer is informative, \
            relevant, and concise:
            {question} 
    [/INST]""",
)
DEFAULT_SEARCH_PROMPT = PromptTemplate(
    input_variables=["question"],
    template="""You are a helpful assistant eager to assist with providing better Google search results. \
        Provide an answer to the following question in about 150 words. Ensure that the answer is informative, \
        relevant, and concise: \
        {question}""",
)
QUESTION_PROMPT_SELECTOR = ConditionalPromptSelector(
    default_prompt=DEFAULT_SEARCH_PROMPT,
    conditionals=[(lambda llm: isinstance(llm, LlamaCpp), DEFAULT_LLAMA_SEARCH_PROMPT)],
)
prompt = QUESTION_PROMPT_SELECTOR.get_prompt(llm)
prompt


"""
Create an LLMChain that connects the selected prompt with the loaded Llama model.
Pass a question into the chain to generate an answer using the LLM.
Invoke the chain with the input dictionary and return the model’s response.
"""
llm_chain = LLMChain(prompt=prompt, llm=llm)
question = "Tell me about Nowshad's experience in Huawei?"
llm_chain.invoke({"question": question})

"""
Convert the Chroma vector store into a retriever that performs similarity search.
Build a RetrievalQA chain that feeds retrieved context into the Llama model using the “stuff” method.
The chain automatically fetches relevant document chunks and injects them into the LLM prompt.
Enable verbose mode to display retrieval steps and model reasoning during execution.
Define a natural‑language query asking about Nowshad Ruhani.
Invoke the QA chain to generate an answer based on both the documents and the LLM.
"""
retriever = vectordb.as_retriever()
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", 
    retriever=retriever, 
    verbose=True
)
query = "Tell me about Nowshad Ruhani?"
qa.invoke(query)

Loading document...
Loaded 1 documents.
Total chunks created: 12


python(40326) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

llama_model_load_from_file_impl: using device MTL0 (Apple M2) (unknown id) - 1619 MiB free
llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from /Users/nowshad/.cache/huggingface/hub/models--TheBloke--Llama-2-7B-Chat-GGUF/snapshots/191239b3e26b2882fb562ffccdd1cf0f65402adb/llama-2-7b-chat.Q2_K.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_len

Embeddings created and stored in ChromaDB. <langchain_community.vectorstores.chroma.Chroma object at 0x10b61a900>


...

Llama.generate: 1 prefix-match hit, remaining 159 prompt tokens to eval


 Nowshad Ruhani is an AI and software engineer with experience in developing and implementing machine learning models and large language models, backend systems, and cloud infrastructure.




llama_perf_context_print:        load time =   17072.83 ms
llama_perf_context_print: prompt eval time =    2024.32 ms /   159 tokens (   12.73 ms per token,    78.55 tokens per second)
llama_perf_context_print:        eval time =    5408.96 ms /    38 runs   (  142.34 ms per token,     7.03 tokens per second)
llama_perf_context_print:       total time =    8688.79 ms /   197 tokens
llama_perf_context_print:    graphs reused =         37



> Finished chain.


{'query': 'Tell me about Nowshad Ruhani?',
 'result': ' Nowshad Ruhani is an AI and software engineer with experience in developing and implementing machine learning models and large language models, backend systems, and cloud infrastructure.\n\n\n'}